# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Comprendre la **Lazy Evaluation**, distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Nettoyer, typer et dédoublonner les tables de la couche **Bronze**.
3. Construire la table analytique enrichie **`/lakehouse/silver/orders`** en optimisant les jointures (**Broadcast Hash Join** vs **SortMergeJoin**).
4. Pratiquer les garanties **ACID de Delta Lake** : inspection du `_delta_log`, **Schema Enforcement**, **Upsert (`MERGE INTO`)** et **Time Travel (`VERSION AS OF`)**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI disponible sur http://localhost:4040")

## 1. Lecture des tables Bronze et Diagnostic Qualité des Données

En couche Bronze, les types ont été inférés depuis le CSV et des anomalies subsistent : valeurs manquantes sur les catégories produits, doublons éventuels, timestamps parfois en chaînes selon les colonnes, avis multiples pour une même commande, ou paiements fractionnés en plusieurs séquences (`credit_card` + `voucher`).

In [ ]:
orders_bz = spark.read.format("delta").load("/lakehouse/bronze/orders")
items_bz = spark.read.format("delta").load("/lakehouse/bronze/order_items")
payments_bz = spark.read.format("delta").load("/lakehouse/bronze/order_payments")
reviews_bz = spark.read.format("delta").load("/lakehouse/bronze/order_reviews")
customers_bz = spark.read.format("delta").load("/lakehouse/bronze/customers")
products_bz = spark.read.format("delta").load("/lakehouse/bronze/products")
cat_trans_bz = spark.read.format("delta").load("/lakehouse/bronze/category_translation")

# Diagnostic des valeurs nulles sur la table orders
null_counts = orders_bz.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"
    ]
])
print("=== Valeurs manquantes dans Bronze `orders` ===")
null_counts.show(truncate=False)

## 2. Nettoyage, Typage Fort et Pré-Agrégations Métier

### Pourquoi pré-agréger `order_items`, `order_payments` et `order_reviews` avant de les joindre à `orders` ?
Une commande (`order_id`) peut contenir **plusieurs articles** (`order_item_id = 1, 2, ...`) ET être réglée via **plusieurs lignes de paiement** (`payment_sequential = 1, 2, ...`).
> **Piège classique de jointure N-à-M :** Joindre directement `order_items` et `order_payments` sur `order_id` crée un **produit cartésien local par commande** qui gonfle artificiellement le chiffre d'affaires !

Nous allons :
1. Enrichir `products` avec la traduction anglaise de catégorie via un **Broadcast Join** (`F.broadcast(cat_trans)`), car la table `category_translation` ne fait que ~70 lignes -> zéro Shuffle réseau !
2. Agréger les articles, paiements et avis à la granularité `order_id` (et sauvegarder également la table fine `/lakehouse/silver/order_items`).

In [ ]:
# 1. Dimension Produits nettoyée avec Broadcast Join sur la table de traduction
products_clean = (
    products_bz
    .drop("_ingested_at", "_source_file")
    .dropDuplicates(["product_id"])
    .join(
        F.broadcast(cat_trans_bz.select("product_category_name", "product_category_name_english")),
        on="product_category_name",
        how="left",
    )
    .withColumn(
        "category",
        F.coalesce(F.col("product_category_name_english"), F.col("product_category_name"), F.lit("unknown")),
    )
)

# Vérifions dans le plan physique que Spark utilise bien BroadcastHashJoin (aucun Exchange/Shuffle sur cat_trans) !
products_clean.explain("simple")

In [ ]:
# 2. Table Silver détaillée `order_items` (filtrage des prix aberrants <= 0)
silver_items = (
    items_bz
    .filter((F.col("price") > 0) & (F.col("freight_value") >= 0))
    .dropDuplicates(["order_id", "order_item_id"])
    .join(
        F.broadcast(products_clean.select("product_id", "category", "product_weight_g")),
        on="product_id",
        how="left",
    )
    .withColumn("category", F.coalesce(F.col("category"), F.lit("unknown")))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
    .withColumn("line_total", F.round(F.col("price") + F.col("freight_value"), 2))
    .select(
        "order_id", "order_item_id", "product_id", "seller_id",
        "category", "product_weight_g", "price", "freight_value", "line_total"
    )
)

(
    silver_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/lakehouse/silver/order_items")
)
print(f"Table /lakehouse/silver/order_items écrite : {silver_items.count():,} lignes")

In [ ]:
# 3. Agrégations à la maille `order_id` pour éviter toute duplication N-M
items_per_order = (
    silver_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("total_items"),
        F.round(F.sum("price"), 2).alias("items_amount"),
        F.round(F.sum("freight_value"), 2).alias("freight_amount"),
        F.round(F.sum("line_total"), 2).alias("total_amount"),
        F.first("category", ignorenulls=True).alias("category"),
    )
)

payments_per_order = (
    payments_bz
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("paid_amount"),
        F.max("payment_installments").alias("max_installments"),
        F.first("payment_type", ignorenulls=True).alias("primary_payment_type"),
    )
)

reviews_per_order = (
    reviews_bz
    .filter(F.col("review_score").between(1, 5))
    .groupBy("order_id")
    .agg(F.round(F.avg("review_score"), 2).alias("review_score"))
)

customers_clean = (
    customers_bz
    .dropDuplicates(["customer_id"])
    .select("customer_id", "customer_unique_id", "customer_city", "customer_state")
)

## 3. Assemblage de la Table Maîtresse `/lakehouse/silver/orders`

Nous calculons maintenant les métriques logistiques clés avec précision horaire convertie en jours décimaux :
- `delivery_days` : délai réel entre l'achat (`order_purchase_timestamp`) et la remise au client (`order_delivered_customer_date`).
- `delay_days` : écart entre la date réelle de livraison et la date promise (`order_estimated_delivery_date`). Positif = commande en retard.
- `is_late` : indicateur binaire (`1` si livrée après la date estimée, `0` sinon).

Ouvrez `http://localhost:4040` (onglets **SQL / DataFrame** et **Stages**) pendant l'exécution de la cellule suivante pour visualiser les échanges **Shuffle** des jointures !

In [ ]:
silver_orders = (
    orders_bz
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn("order_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .withColumn(
        "delivery_days",
        F.round(
            (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400.0,
            2,
        ),
    )
    .withColumn(
        "delay_days",
        F.round(
            (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_estimated_delivery_date")) / 86400.0,
            2,
        ),
    )
    .withColumn(
        "is_late",
        F.when(F.col("delay_days") > 0, F.lit(1)).otherwise(F.lit(0)),
    )
    # Nettoyage des anomalies temporelles éventuelles (livraison antérieure à l'achat)
    .filter(F.col("delivery_days").isNull() | (F.col("delivery_days") >= 0))
    .join(customers_clean, on="customer_id", how="left")
    .join(items_per_order, on="order_id", how="inner")
    .join(payments_per_order, on="order_id", how="left")
    .join(reviews_per_order, on="order_id", how="left")
    .withColumn("category", F.coalesce(F.col("category"), F.lit("unknown")))
)

(
    silver_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/lakehouse/silver/orders")
)

print(f"Table Delta /lakehouse/silver/orders créée avec succès : {silver_orders.count():,} commandes !")
silver_orders.select(
    "order_id", "customer_state", "category", "order_status",
    "total_amount", "delivery_days", "delay_days", "is_late", "review_score"
).show(8, truncate=False)

## 4. Exploration des Garanties ACID de Delta Lake

### 4.1 Anatomie du journal transactionnel `_delta_log` sur HDFS
Contrairement à un dossier Parquet classique où une écriture interrompue laisse des fichiers corrompus visibles des lecteurs, Delta Lake enregistre chaque transaction sous forme d'un commit JSON atomique dans `_delta_log/00000000000000000000.json`.

In [ ]:
# Lecture directe du premier fichier JSON de commit Delta sur HDFS
delta_commits = spark.read.text("/lakehouse/silver/orders/_delta_log/00000000000000000000.json")
delta_commits.show(10, truncate=110)

### 4.2 Schema Enforcement (Protection contre la pollution de schéma)
Tentons volontairement d'ajouter (`append`) une ligne avec des colonnes incompatibles dans `/lakehouse/silver/orders`.

In [ ]:
bad_df = spark.createDataFrame([("ord_fake_1", "colonne_intruse", "pas_un_montant")], ["order_id", "intruder_col", "total_amount"])

try:
    bad_df.write.format("delta").mode("append").save("/lakehouse/silver/orders")
except Exception as e:
    print("Schema Enforcement activé ! Écriture rejetée proprement par Delta Lake :")
    print(" ->", str(e).splitlines()[0])

### 4.3 Upsert Transactionnel (`MERGE INTO`) et Time Travel (`VERSION AS OF`)
En production, les commandes `shipped` finissent par être livrées (`delivered`) le lendemain. Avec Parquet pur, il faudrait réécrire toute la table. Avec Delta Lake, nous exécutons un `MERGE INTO` ciblé, puis nous vérifions que nous pouvons toujours interroger l'état antérieur de la table grâce au **Time Travel** !

In [ ]:
# Sélectionnons jusqu'à 5 commandes 'shipped' pour simuler leur confirmation de livraison
shipped_sample = (
    spark.read.format("delta").load("/lakehouse/silver/orders")
    .filter(F.col("order_status") == "shipped")
    .limit(5)
    .select(
        "order_id",
        F.lit("delivered").alias("new_status"),
        F.lit(12.5).alias("new_delivery_days"),
        F.lit(0).alias("new_is_late"),
    )
)

nb_updates = shipped_sample.count()
print(f"Lot de mise à jour préparé : {nb_updates} commande(s) passent de 'shipped' à 'delivered'.")

if nb_updates > 0:
    target_delta = DeltaTable.forPath(spark, "/lakehouse/silver/orders")
    (
        target_delta.alias("target")
        .merge(
            shipped_sample.alias("updates"),
            "target.order_id = updates.order_id",
        )
        .whenMatchedUpdate(set={
            "order_status": "updates.new_status",
            "delivery_days": "updates.new_delivery_days",
            "is_late": "updates.new_is_late",
        })
        .execute()
    )
    print("MERGE INTO exécuté avec succès !")

In [ ]:
# Historique complet des transactions ACID et comparaison Time Travel (Version 0 vs Version courante)
spark.sql("DESCRIBE HISTORY delta.`/lakehouse/silver/orders`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=80)

v0_shipped = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load("/lakehouse/silver/orders")
    .filter(F.col("order_status") == "shipped")
    .count()
)
latest_shipped = (
    spark.read.format("delta")
    .load("/lakehouse/silver/orders")
    .filter(F.col("order_status") == "shipped")
    .count()
)

print(f"Commandes 'shipped' en VERSION AS OF 0 : {v0_shipped:,}")
print(f"Commandes 'shipped' en VERSION COURANTE : {latest_shipped:,} (écart = {v0_shipped - latest_shipped})")

### Bilan du Jour 2
- Vous avez construit `/lakehouse/silver/orders` et `/lakehouse/silver/order_items` sans produit cartésien accidentel et en exploitant le **Broadcast Join**.
- Vous savez auditer `_delta_log`, effectuer un `MERGE INTO` transactionnel et voyager dans le temps avec `versionAsOf`.
- Passez maintenant au notebook **`J3_Analytics_SQL_Gold.ipynb`** pour produire les indicateurs décisionnels de la couche **Gold** !